# Solving the Burgers Equation Using a PINN

In this notebook we are going to implement the continuous time model of the Physics-Informed Neural Networks, which it was presented in Raissi, Perdikaris & Karniadakis (2019), *Journal of Computational Physics 378*, 686–707. In particular, we are going to reproduce the Burgers Equation example.

The idea is representate the $u(x, y)$ using a neural network $u_{\theta}(x, y)$. Through automatic differentation we derivate the network with regard its inputs $(t, x)$ and we calculate the EDP residual $f(t, x)$. To train this neural network we are going to minimize the loss against the initial conditions and the EDP residual in collocation points.

So we are considering the Burgers equation given by:
$$
    u_t + u\,u_x - \nu\,u_{xx} = 0, \qquad x\in[-1,1],\; t\in[0,1], \qquad \nu = \frac{0.01}{\pi},
$$
with the initial and boundary condition:
$$
    u(0,x) = -\sin(\pi x), \qquad u(t,-1) = u(t,1) = 0.
$$
The EDP residual is defenied simply by:
$$
    f(t, x) := u_t + u\,u_x - \nu\,u_{xx},
$$
and the loss function is:
$$
    \mathcal{L}_{MSE} = \mathcal{L}_{MSE_{u}} + \mathcal{L}_{MSE_{f}},
$$
where
$$
    \mathcal{L}_{MSE_{u}} = \frac{1}{N_u}\sum_{i=1}^{N_u}\left|u(t_u^i,x_u^i)-u^i\right|^2, \qquad \mathcal{L}_{MSE_{f}} = \frac{1}{N_f}\sum_{i=1}^{N_f}\left|f(t_f^i,x_f^i)\right|^2.
$$

## 1. Initial Configuration

Here we are only importing the basic libraries and the modules of the project which are in `src/`.

In [ ]:
# To import my files
import sys
import os

src = os.path.abspath("./src")

if src not in sys.path:
    sys.path.append(src)

In [ ]:
# Imports
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
from matplotlib import pyplot as plt

from src.model import PINNSolution
from src.physics import compute_pde_residual
from src.dataset import generate_domain_samples

## 2. Experiment Parameters

In this code we are considering de escential parameters of the project: `VISCOSITY` is the $\nu$ coefficient; `X_MIN`, `X_MAX` and `T_MIN`, `T_MAX` are just the spatial-temporal domain those are $[-1, 1]$ and $[0, 1]$ respectively; `N_U` and `N_F` are the number of train points for conditions and collocation points, respectively; `LAYERS` is a list with the neurons number for each layer in the neural network; we are using an Adam with `LEARNING_RATE` of $10^{-3}$ and trained during 2000 `EPOCHS`; lastly, we are training the model in `DEVICE`.

In [ ]:
# Parameters of the exp.

# Physics parameters
VISCOSITY = 0.01 / torch.pi # Burgers equation

# Domain
X_MIN, X_MAX = -1.0, 1.0
T_MIN, T_MAX = 0.0, 1.0

# Number of train points
N_U = 100 # Initial and boundary conditions
N_F = 10000 # Collocation points

# Neural network architecture
LAYERS = [2, 20, 20, 20, 20, 20, 20, 20, 20, 1]

# Optimization hyperparameters
LEARNING_RATE = 1e-3
EPOCHS = 2000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

This code is only to fix a seed.

In [ ]:
# Fix seed
torch.manual_seed(151)

## 3. Train Data

Here we have the `generate_domain_samples` which generates two points sets: `(t_u, x_u, u_u)` come from the initial and boundary conditions. On the other hand, `(t_f, x_f)` come from the collocation points which are distributed through the entire domain.

In [ ]:
# Generate data
(t_u, x_u, u_u), (t_f, x_f) = generate_domain_samples(
    N_U, N_F, X_MIN, X_MAX, T_MIN, T_MAX, DEVICE
)

## 4. Model, Optimizer Alghorithm and Train Loop

In this code we instance the model `PINNSolution` which is going to be trained with the Adam optimizer and we use the `MSELoss` to calculate $\mathcal{L}_{MSE_u}$ and $\mathcal{L}_{MSE_f}$.

In [ ]:
# Model
model = PINNSolution(LAYERS).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
mse_loss = nn.MSELoss()

loss_history = []

This is basically the train loop. Here it is only interesting to know that if you use a cpu this code could take roughly 2 minutes.

In [ ]:
# Train loop
model.train()

for _ in tqdm(range(1, EPOCHS + 1), desc="Train progress", unit=" Epoch"):
    optimizer.zero_grad()
    
    u_pred = model(t_u, x_u)
    loss_u = mse_loss(u_pred, u_u)
    
    f_pred = compute_pde_residual(model, t_f, x_f, VISCOSITY)
    loss_f = mse_loss(f_pred, torch.zeros_like(f_pred))
    
    loss = loss_u + loss_f
    loss.backward()
    optimizer.step()
    
    loss_history.append(loss.item())

## Results

First, before looking at the charts, we need to create a grid, in this case $200 \times 200$ to representate the space $(t, x)$.

In [ ]:
model.eval()

t = np.linspace(T_MIN, T_MAX, 200)
x = np.linspace(X_MIN, X_MAX, 200)
T, X = np.meshgrid(t, x)

t_tensor = torch.tensor(T.flatten()[:, None], dtype=torch.float32).to(DEVICE)
x_tensor = torch.tensor(X.flatten()[:, None], dtype=torch.float32).to(DEVICE)

with torch.no_grad():
    u_pred = model(t_tensor, x_tensor).cpu().numpy()
    
U = u_pred.reshape(T.shape)

We can show the heat map, as this implementation is doing its job very well. This is the expected behavior, if we considering the original paper results.

In [ ]:
plt.figure(figsize=(11, 4))

contour = plt.pcolormesh(T, X, U, cmap="rainbow", shading="auto")

plt.colorbar(contour, label='$u(t, x)$')

plt.xlabel('Time ($t$)')
plt.ylabel('Space ($x$)')
plt.tight_layout()

plt.show()

Here we can show the total loss $\mathcal{L}_{MSE_u} + \mathcal{L}_{MSE_f}$ per epoch, in logarithm scale.

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(loss_history, label='Total Loss')
plt.yscale('log')

plt.xlabel('Iterations')
plt.ylabel('MSE (Log Scale)')

plt.grid(True, which="both", ls="--")

plt.legend()
plt.tight_layout()

plt.show()